In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
#Import path to work with file and folder paths
from pathlib import Path

#Import pandas to read and manipualte tabular data
import pandas as pd

#Import tarfile to open and extract the compressed zip archive
import tarfile

#Import urllib.request to download files from the url
import urllib.request

def load_housing_data():
    #Defining path where the data will be stored
    tarball_path = Path("datasets/housing.tgz")

    #Download the data only if it doesn't exist
    if not tarball_path.is_file():

        #Create the dataset directory if it doesnot exist
        Path("datasets").mkdir(parents=True, exist_ok=True)

        #URL of the housing dataset
        url = "https://github.com/ageron/data/raw/main/housing.tgz"

        #Download the archive and save it to the tarbal_path
        urllib.request.urlretrieve(url, tarball_path)

    #Open the archive and close it automatically after use
    with tarfile.open(tarball_path) as housing_tarball:

        #Extract the archives content in to the dataset directory
        housing_tarball.extractall(path="datasets")

    #Read the extracted csv file into the panads dataframe and return it
    return pd.read_csv(Path("datasets/housing/housing.csv"))

housing = load_housing_data()
housing.head()

In [ ]:
#A quick description of the data, in particular
#total number of rows and each attribute's type and number of non-null values
housing.info()

In [ ]:
#You can find out what categories exist and how many districts belong to each category by using the value_counts() method:
housing["ocean_proximity"].value_counts()

In [ ]:
#This shows the summary of the numerical attributes
housing.describe()

In [ ]:
#This code helps us to save figures

IMAGES_PATH = Path() /"images" / "end_to_end_project"
IMAGES_PATH.mkdir(parents=True, exist_ok=True)

def save_fig(fig_id, tight_layout=True, fig_extension = "png", resolution=300):
        path = IMAGES_PATH / f"{fig_id}.{fig_extension}"
        if tight_layout:
                plt.tight_layout()
        plt.savefig(path, format=fig_extension, dpi=resolution)


In [ ]:
#Notice that the dataset is scaled and capped. 
#The datas written are not the actual value but a representation of the actual value.
#Capping means putting a maximum or minimum limit on the recorded values.
#Scaling means the original income values were transformed into smaller numbers.
import matplotlib.pyplot as plt

plt.rc('font', size=14)
plt.rc('axes', labelsize=14, titlesize=14)
plt.rc('legend', fontsize='14')
plt.rc('xtick', labelsize=10)
plt.rc('ytick', labelsize=10)

housing.hist(bins=50, figsize=(12,8))
save_fig("attribute_histogram_plots")
plt.show()

# **Test Set**

In [ ]:
#Import numpy for numerical operations and random shuffling
import numpy as np


def shuffle_and_split_data(data, test_ratio):
    #Create a list of all row positions and put them in random order
    shuffled_indices = np.random.permutation(len(data))

    #Calculate how many rows should be used for testing
    test_set_size = int(len(data)* test_ratio)

    #Take the first group for random row positions for testing
    test_indices = shuffled_indices[:test_set_size]

    #Take the remaining random row positions for training
    train_indices = shuffled_indices[test_set_size:]

    #Use the selected row positions to separate the data into training
    #Return the training set first and test set second
    return data.iloc[train_indices], data.iloc[test_indices]
    

In [ ]:
#This shiffles and splits the dataset then outputs the number of rows in the training set
train_set, test_set = shuffle_and_split_data(housing, 0.2)
len(train_set)

In [ ]:
len(test_set)

In [ ]:
np.random.seed(42)

* Randomness makes your test set change everytime you run the code.
* If you run the program gain, it will generate a different test set. Over time you will get to see the whole dataset which you don't want.
  Solution
1. Save the test set on the first run and then load it to subsequent runs.
2. Or Set random number generator's seed before calling np.random.permutation(), so that it always generate the same shuffled indices 

**Reproducability**
* When you repeat an exdperiment under the same conditions, you can obtain the same result.
* Instead of relying on a random seed, yo ucan save the selected test rows and load them again later.
* This is useful when you need to the exact same split across different scripts or environments.
* Another approach is to use a stable identifer for each row to determine whether it belongs to the training or test set.
* Stratified sampling: Splitting data so that improtant groups are represented in approxiamtely the same proportions as in the original dataset.

**Splitting the test and training set correctly**
1. Stable splitting: Keep the same records in the test set even when new data arrives.
2. Random vs Stratified sampling: Random can accidentally misrepresent important groups; stratified sampling preserves their proportions
3. Income categories: Because median income matters when predicting hous prices, we create income groups and ensure each group is represented appropriately in the test set.

In [ ]:
from zlib import crc32

def is_id_in_test_set(identifier, test_ratio):
    return crc32(np.int64(identifier)) < test_ratio * 2**32

def split_data_with_id_hash(data, test_ratio, id_column):
    ids = data[id_column]
    in_test_set = ids.apply(lambda id_: is_id_in_test_set(id_, test_ratio))
    return data.loc[~in_test_set], data.loc[in_test_set]

housing_with_id = housing.reset_index()  # Adds an index column
train_set, test_set = split_data_with_id_hash(housing_with_id, 0.2, "index")
housing_with_id['id'] = housing["longitude"] * 1000 + housing["latitude"]
train_set, test_set = split_data_with_id_hash(housing_with_id, 0.2, "id")

In [ ]:
from sklearn.model_selection import train_test_split

train_set, test_set = train_test_split(housing, test_size=0.2, random_state=42)

In [ ]:
test_set["total_bedrooms"].isnull().sum()

**Binomial distrbution**
* To find the probablity that a random sample of 1000 people contains less than 48.5% female or more than 53.5% female when the popluation's female ration is 51.1%.
* The cdf() method of the binomail distrbution gives us the probablity that the number of females will be equal or less than the given value.
 1. Sample size(n) =1000
 2. Population proportion(p): 51.1%
 3. Target proportions: < 48.5% || > 53.5%
 4.  Mean = n x p = 1000 x 0.511 = 511
 5.  Standard deviation: (n x p x (1-p)) ^ 1/2 = 15.8076
 6.  To be continuedd....

In [ ]:
from scipy.stats import binom

sample_size = 1000
ratio_female = 0.511
proba_too_small = binom(sample_size, ratio_female).cdf(485-1)
proba_too_large= 1- binom(sample_size, ratio_female).cdf(535)
print(proba_too_small + proba_too_large)

# **Visualizing Geographical Data**

Here we uncover:
* Spatial Clusters
* Detect non-linear geolocation patterns
* Multi_variable interaction analysis
* Guide feature engineering: seeing spatial relationships directly siggests meaningful new features to engineer.
* **Once we visualize the data and spot patterns, we turn those visual insights into inputs a machine learning model can actually use.**

In [ ]:
housing.plot(kind="scatter", x="longitude", y="latitude" grid=True)
plt.show()

In [ ]:
housing.plot(kind ="scatter", x="longitude", y="latitude", grid=True, alpha=0.2)
plt.show()

In [ ]:
housing.plot(kind="scatter", x="longitude", y="latitude", grid=True, 
            s=housing["population"]/100, label="population",
            c="median_house_value", cmap="jet", colorbar=True,
            legend=True, sharex=False, figsize=(10,7))
plt.show()

# **Looking for correlations**
* It is used to check the correlation matrix(corr()) againist your target variable(median_house_value) is to understand how strongly each feature related linearly to what you want to predict.
* It helps us identify key predictors.
* 

In [ ]:
filename = "california.png"


# **Handling text and categorical attributes**
* The ocean_proximity is left out earlier because it is a text attribute and we can not compute its median.
* Most machine learning algorithms prefer to work with numbers, so let's convert text categories from text numbers.

In [ ]:
housing_cat = housing[["ocean_proximity"]]
housing_cat.head(8)

In [ ]:
##Converting the text attribute to number so the machine will understand
from sklearn.preprocessing import OrdinalEncoder

ordinal_encoder = OrdinalEncoder()
housing_cat_encoded = ordinal_encoder.fit_transform(housing_cat)
housing_cat_encoded[:8]

**One issue here with this representation of ML algorithms will assume that two nearby values are more similar than two distant values.**
* To fix this issue, a common solution is to create on binary attribute per category: one categiry equal to 1 when the category is "<1H OCEAN" or 0 otherwise and this is called One-hot encoding.
* **One hot encoding is** a data processing technique that convert categorical varaibles in to 0 and 1.
* Only one attribute will be equal to 1 (hot) while others will be 0(Cold).
* The new attributes are sometimes called dummy attributes.

In [ ]:
from sklearn.preprocessing import OneHotEncoder
cat_encoder = OneHotEncoder()
housing_cat_1hot = cat_encoder.fit_transform(housing_cat)

In [ ]:
housing_cat_1hot
## The output of this is a SciPy sparse matrix.
# This is very useful since most of the numbers are just zeros.
# This will try to save up memory, and only save the exact location of 1s.

In [ ]:
#If you really want the NumPy array...
housing_cat_1hot.toarray()

* One-hot encoding creates too many columns for high-cardinality categories, wasting memory and slowing down model training.
• The Fix: Replace categories with specific numerical facts (like distance or GDP) or use compact, learned numeric vectors called embeddings - which are compact, fixed-size lists of numbers learned by a model that capture the real-world meaning and relationships of categories.

# **Custom Transformers**
* Although Scikit-Learn provides many useful transformers, you will need to write your own for tasks such as custom cleanup operations or combining specific attributes. You will want your transformer to work seamlessly with Scikit-Learn func‐
tionalities (such as pipelines), and since Scikit-Learn relies on duck typing (not inheritance), all you need is to create a class and implement three methods:**fit()(returning self), transform(), and fit_transform().**

In [ ]:
# A simple transformer class that adds the combined attributes we discussed earlier
from sklearn.base import BaseEstimator, TransformerMixin

rooms_ix, bedrooms_ix, population_ix, households_ix =3, 4, 5, 6

class CombinedAttributesAdder(BaseEstimator, TransformerMixin):
    def _init_(self, add_bedrooms_per_roon = True)
        self.add_bedrooms_per_room = add_bedrooms_per_room
    def fit(self, X, y=None):
        return self
    def transform(self, X, y=None):
        rooms_per_household = X[:, rooms_ix] / [X: households_ix]
        population_per_household = X[:, population_ix] / X[:, households_ix]
         if self.add_bedrooms_per_room:
             bedrooms_per_room = X[:, bedrooms_ix] / X[:, rooms_ix]
             return np.c_[X, rooms_per_household, population_per_household,bedrooms_per_room]
         else:
             return np.c_[X, rooms_per_household, population_per_household]
attr_adder = CombinedAttributesAdder(add_bedrooms_per_room=False)
housing_extra_attribs = attr.adder.transform(housing.values)

* In the above code: the transformer had one hyperparameter, add_bedrooms_per_room. This hyperparameter will allow you to easily fins out whether adding this attribute helps the Machine learning algorthms or not.
* More generally, you can add a parameter to gate any data preparation step that you are not 100% sure about. The more you automate these data prep steps, the more combinations you can automaticallt try our, making it much more likely that you will find a great combination.

# **Feature Scaling**
* ML algorithms don't perform well when the input numerical attributes have very different scales.
* There are two common ways to get all attributes to have the same scale: **min-max scaling** and **standardization**
* **Min Max scaling(Normalization)**: values are shifted and rescaled so that they endup ranging from 0 to 1.
  * Subtract the min value
  * Divide by (max - min)
    * Scikit-learn provides a transformer called *MinMaxScaler*.
    * It has a feature_range hyperparameter that lets you change the range if you don't want 0-1 for some reason.
* **Standardization**: It subtracts the mean value, then it divides by the standard deviation so that the resulting distrbution has unit variance.
    * It doesn't have a specific range like Normalization which makes it hard for neural networks
    * It is much less affected by outliers.
    * It provides a transformer called *StandrardScaler* for standardization.

# **Transformation pipelines**
* There are data transformation steps that need to be executed in the right order.
* Sckit learn provides the pipeline class to help with such sequences of transfromations.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
num_pipeline = Pipeline([
 ('imputer', SimpleImputer(strategy="median")),
 ('attribs_adder', CombinedAttributesAdder()),
 ('std_scaler', StandardScaler()),
 ])
housing_num_tr = num_pipeline.fit_transform(housing_num)

* The pipeline constructor takes a list of name/estimator pairs defining a sequence of steps.
* It automates and chains together your machine learning steps into a single, clean workflow.
  1. It takes a list of names and parameters.
  2. All but the last estimator must be transformers. Must have a fit_transform() method. Transformers are just data cleaners.It looks at the data, learn from it, and modify it—like filling in missing values or scaling numbers
  3. The very last item in the pipeline is usually your actual ML model(Regression model, Random Forest)
  4. When you pass your raw training data into pipeline.fit(), Scikit-Learn automatically triggers a domino effect behind the scenes:
Raw Data 
   │
   ▼
[ Step 1: "scaler" ] ──► Calls .fit_transform() ──► Outputs Scaled Data
              │
   ┌──────────────────────────────────────────────────────┘
   ▼
[Step 2: "imputer"] ──► Calls .fit_transform() ──► Outputs Cleaned & Scaled Data
                                                          │
   ┌──────────────────────────────────────────────────────┘
   ▼
[ Final Step: "model" ] ──► Calls ONLY .fit() ─────► Model is now trained!

 It is all a process of cleaning and scaling it after learning it till it hits the last step where the learning model exists.

* The pipeline exposes the same methods as the final estimator. In the above example, the last estimator is a StandardScaler, which is a transformer. So the pipeline has a transform method that applies all the transforms to the data in sequence.
* Sicne we don't want to handle categorical columns and numerical columns separately anymore we are going to use the columntransform() method.

In [ ]:
from sklearn.compose import ColumnTransformer
num_attribs = list(housing_num)
cat_attribs = ["ocean_proximity"]
full_pipeline = ColumnTransformer([
 ("num", num_pipeline, num_attribs),
 ("cat", OneHotEncoder(), cat_attribs),
 ])
housing_prepared = full_pipeline.fit_transform(housing)

1. We import the ColumnTransformer class.
2. We get the
list of numerical column names and the list of categorical column names, and we
construct a ColumnTransformer.
* The constructor requires a list of tuples, where each
tuple contains a name, a transformer and a list of names (or indices) of columns
that the transformer should be applied to.
* One hot encoder returns a sparse matrix whilethe num_pipleine returns a dense matrix.
* Instead of a transformer, you can specify the string "drop" if you want the columns to be dropped. Or you can specify "pass through" if you want the columns to be left untouched. By default,
the remaining columns (i.e., the ones that were not listed) will be dropped, but you can set the remainder hyperparameter to any
transformer (or to "passthrough") if you want these columns to be handled differently

# **SELECT AND TRAIN THE MODEL**


In [ ]:
# Selecting and training linear regression model
from sklearn.linear_model import LinearRegression

lin_reg = LinearRegression()
lin_reg.fit(housing_prepared, housing_labels)

In [ ]:
# Measuring the regression model RMSE on the whole training set using sckit learn's RMSE
>>> from sklearn.metrics import mean_squared_error
>>> housing_predictions = lin_reg.predict(housing_prepared)
>>> lin_mse = mean_squared_error(housing_labels, housing_predictions)
>>> lin_rmse = np.sqrt(lin_mse)
>>> lin_rmse

In [ ]:
# Trying out descision regressor since linear regression generated the error we don't want
from sklearn.tree import DecisionTreeRegressor
tree_reg = DecisionTreeRegressor()
tree_reg.fit(housing_prepared, housing_labels)

In [ ]:
# Testing the decision regressor on our training set
>>> housing_predictions = tree_reg.predict(housing_prepared)
>>> tree_mse = mean_squared_error(housing_labels, housing_predictions)
>>> tree_rmse = np.sqrt(tree_mse)
>>> tree_rmse

In [ ]:
#Try it out on a few instances from the training set.
>>> some_data = housing.iloc[:5]
>>> some_labels = housing_labels.iloc[:5]
>>> some_data_prepared = full_pipeline.transform(some_data)
>>> print("Predictions:", lin_reg.predict(some_data_prepared))
Predictions: [ 210644.6045 317768.8069 210956.4333 59218.9888 189747.5584]
>>> print("Labels:", list(some_labels))
Labels: [286600.0, 340600.0, 196900.0, 46300.0, 254500.0]

# **Better Evaluation using cross validation**

* The decision regressor generated 0 error. So we must test is using a model validation set.
* We will evaluate it using train_test_split function.
* As an alternative we can use Sckit_Learn's K-fold cross_validation feature.
* The next code randomly splits the training set into 10 district subsets called folds, then it trains and evaluated the decision tree model 10 times, picking a different fold for evaluation everytime and training on the other 9 folds. 

In [ ]:
from sklearn.model_selection import cross_val_score
scores = cross_val_score(tree_reg, housing_prepared, housing_labels,
scoring="neg_mean_squared_error", cv=10)
tree_rmse_scores = np.sqrt(-scores)

In [ ]:
#Our result
>>> def display_scores(scores):
... print("Scores:", scores)
... print("Mean:", scores.mean())
... print("Standard deviation:", scores.std())
...
>>> display_scores(tree_rmse_scores)

In [ ]:
#Computing the same scored for the linear regression model
>>> lin_scores = cross_val_score(lin_reg, housing_prepared, housing_labels,
... scoring="neg_mean_squared_error", cv=10)
...
>>> lin_rmse_scores = np.sqrt(-lin_scores)
>>> display_scores(lin_rmse_scores)